# Latent profiles: prompt hand-test (Week 1)

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On** (needs a phone-verified Kaggle account).

Then *Run All*. Takes ~15-25 min, mostly installing vLLM and downloading the model. At the end, download `handtest_outputs.zip` from the Output panel and put it in the project folder.

Generated by `scripts/build_kaggle_notebook.py`; don't edit by hand.

In [ ]:
!pip install -q vllm sentence-transformers huggingface_hub
!pip uninstall -y -q torchaudio
import torch; print(torch.__version__, torch.cuda.device_count(), 'GPUs')

In [ ]:
import os
for d in sorted({os.path.dirname(f) for f in ['src/obliq/__init__.py', 'src/obliq/data.py', 'src/obliq/runs.py', 'src/obliq/profiles.py', 'scripts/handtest_prompts.py', 'prompts/math_problem.txt', 'prompts/math_problem_solution.txt', 'prompts/writing_style.txt']}):
    os.makedirs(f'/kaggle/working/irproj/{d}', exist_ok=True)
%cd /kaggle/working/irproj

In [ ]:
%%writefile src/obliq/__init__.py
# package marker


In [ ]:
%%writefile src/obliq/data.py
"""Load OBLIQ-Bench tasks (corpus, queries, gold/pooled qrels, per-query exclusions)."""
import csv
import json
from dataclasses import dataclass, field
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
DATA = ROOT / "data" / "obliq"
TASK_DIRS = {"math": "analogues/math", "writing": "analogues/writing"}

Qrels = dict[str, dict[str, int]]


@dataclass
class Task:
    name: str
    corpus: dict[str, str]
    queries: dict[str, str]
    qrels: Qrels                      # gold judgments
    qrels_pool: Qrels | None          # pooled judgments (Math only)
    excluded: dict[str, set[str]] = field(default_factory=dict)


def download(tasks=("math", "writing")):
    from huggingface_hub import snapshot_download
    snapshot_download("dianetc/OBLIQ-Bench", repo_type="dataset", local_dir=DATA,
                      allow_patterns=[f"{TASK_DIRS[t]}/*" for t in tasks] + ["README.md"])


def _read_jsonl(path: Path) -> dict[str, str]:
    with open(path, encoding="utf-8") as f:
        return {d["_id"]: d["text"] for d in map(json.loads, f) if d}


def _read_qrels(path: Path) -> Qrels | None:
    # Tab-separated with a header; column names differ per task (query-id vs query_id).
    if not path.exists():
        return None
    qrels: Qrels = {}
    with open(path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)
        for qid, did, score in reader:
            if int(score) > 0:
                qrels.setdefault(qid, {})[did] = int(score)
    return qrels


def load_task(name: str) -> Task:
    base = DATA / TASK_DIRS[name]
    qdir = base / "queries+qrels"
    if not (base / "corpus" / "corpus.jsonl").exists():
        download((name,))
    excl_path = qdir / "per_query_excluded_ids.json"
    excluded = {}
    if excl_path.exists():
        excluded = {q: set(ids) for q, ids in json.loads(excl_path.read_text()).items()}
    return Task(
        name=name,
        corpus=_read_jsonl(base / "corpus" / "corpus.jsonl"),
        queries=_read_jsonl(qdir / "queries.jsonl"),
        qrels=_read_qrels(qdir / "qrels.tsv"),
        qrels_pool=_read_qrels(qdir / "qrels_pool.tsv"),
        excluded=excluded,
    )


In [ ]:
%%writefile src/obliq/runs.py
"""Run files (qid -> {doc_id: score}), the exclusion mask, and top-k helpers."""
import json
from pathlib import Path

import numpy as np

from .data import ROOT

RUNS = ROOT / "runs"
Run = dict[str, dict[str, float]]


def mask(run: Run, excluded: dict[str, set[str]], k: int | None = None) -> Run:
    """Drop each query's excluded doc ids, then keep the top k. Every system goes through this."""
    out = {}
    for qid, docs in run.items():
        drop = excluded.get(qid, ())
        kept = sorted(((d, s) for d, s in docs.items() if d not in drop), key=lambda x: -x[1])
        out[qid] = dict(kept[:k] if k else kept)
    return out


def topk_from_scores(scores: np.ndarray, doc_ids: list[str], k: int) -> dict[str, float]:
    k = min(k, len(doc_ids))
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return {doc_ids[i]: float(scores[i]) for i in idx}


def run_path(task: str, system: str) -> Path:
    return RUNS / task / f"{system}.json"


def save_run(run: Run, task: str, system: str) -> Path:
    path = run_path(task, system)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(run))
    return path


def load_run(task: str, system: str) -> Run:
    return json.loads(run_path(task, system).read_text())


In [ ]:
%%writefile src/obliq/profiles.py
"""Latent profiles: build profiling prompts, generate with an LLM, cache to disk.

Cache layout: profiles/<task>/<variant>/{docs,queries}.jsonl with {"_id", "profile"} per line,
plus meta.json recording the prompt hash, model and generation settings. A variant's cache is
never mixed with output from a different prompt or model.
"""
import hashlib
import json
import re
from pathlib import Path

from .data import DATA, ROOT, Task

PROMPTS = ROOT / "prompts"
PROFILES = ROOT / "profiles"

MATH_Q_PREFIX = "Given the following mathematical problem:\n\n"
MATH_Q_SUFFIX_START = "\n\nFind other mathematical problems that require the same core reasoning pattern"

# Which prompt each task uses for documents and queries (queries never see solutions).
TASK_PROMPTS = {
    "writing": {"doc": "writing_style", "query": "writing_style"},
    "math": {"doc": "math_problem", "query": "math_problem"},
    "math_soln": {"doc": "math_problem_solution", "query": "math_problem"},
}

MAX_CHARS = 4000  # truncate long Writing snippets; style is visible well before this


def strip_math_query(text: str) -> str:
    """Math queries wrap a problem in fixed boilerplate; profile only the problem."""
    assert text.startswith(MATH_Q_PREFIX), text[:80]
    return text[len(MATH_Q_PREFIX):].split(MATH_Q_SUFFIX_START)[0].strip()


def load_math_solutions() -> dict[str, dict[str, str]]:
    rows = json.loads((DATA / "analogues/math/final_dataset.json").read_text(encoding="utf-8"))
    return {f"{r['competition']}___{r['year']}___{r['problem_id']}": r for r in rows}


def read_prompt(name: str) -> str:
    return (PROMPTS / f"{name}.txt").read_text(encoding="utf-8")


def build_prompts(task: Task, kind: str, ids: list[str], setting: str | None = None) -> dict[str, str]:
    """User-turn prompt text for each id. kind is 'doc' or 'query'."""
    setting = setting or task.name
    template = read_prompt(TASK_PROMPTS[setting][kind])
    if kind == "query":
        src = task.queries
        clean = strip_math_query if task.name == "math" else (lambda t: t)
        return {i: template.format(text=clean(src[i])[:MAX_CHARS]) for i in ids}
    if setting == "math_soln":
        sol = load_math_solutions()
        return {i: template.format(problem=sol[i]["problem"], solution=sol[i]["solution"][:MAX_CHARS])
                for i in ids}
    return {i: template.format(text=task.corpus[i][:MAX_CHARS]) for i in ids}


# ---------------------------------------------------------------- generation backends

class HFBackend:
    """Local transformers backend (Windows-friendly; used for hand-testing on the laptop GPU)."""

    # transformers has no output-only frequency penalty; block long verbatim loops instead.
    decoding = "greedy-norepeat8"

    def __init__(self, model: str, max_new_tokens: int = 220, batch_size: int = 4):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(model, padding_side="left")
        self.model = AutoModelForCausalLM.from_pretrained(model, device_map="cuda:0", torch_dtype=torch.float16)
        self.max_new_tokens, self.batch_size = max_new_tokens, batch_size

    def generate(self, prompts: list[str]) -> list[str]:
        import torch
        out = []
        for b in range(0, len(prompts), self.batch_size):
            chats = [self.tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                                  add_generation_prompt=True)
                     for p in prompts[b:b + self.batch_size]]
            enc = self.tok(chats, return_tensors="pt", padding=True).to(self.model.device)
            with torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=self.max_new_tokens, do_sample=False,
                                          no_repeat_ngram_size=8, pad_token_id=self.tok.pad_token_id)
            out += self.tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        return [o.strip() for o in out]


class VLLMBackend:
    """Batched vLLM backend (Kaggle / Linux GPU) for profiling whole corpora."""

    # Greedy decoding looped on some Writing profiles ("metabolic signatures" x30); a frequency
    # penalty on generated tokens only (not the prompt) stops that without blocking quotes.
    decoding = "greedy-freqpen0.3"

    def __init__(self, model: str, max_new_tokens: int = 220, **llm_kwargs):
        from vllm import LLM, SamplingParams
        # T4s have no bf16; one T4 (16 GB) holds the 7B AWQ model, leaving the 2nd GPU free.
        kw = {"dtype": "half", "max_model_len": 4096, "gpu_memory_utilization": 0.85, "seed": 0}
        self.llm = LLM(model=model, **{**kw, **llm_kwargs})
        self.params = SamplingParams(temperature=0.0, frequency_penalty=0.3, max_tokens=max_new_tokens)

    def generate(self, prompts: list[str]) -> list[str]:
        msgs = [[{"role": "user", "content": p}] for p in prompts]
        return [o.outputs[0].text.strip() for o in self.llm.chat(msgs, self.params, use_tqdm=True)]


# ---------------------------------------------------------------- cache

def _meta(task_setting: str, kind: str, model: str, max_new_tokens: int, decoding: str) -> dict:
    prompt = read_prompt(TASK_PROMPTS[task_setting][kind])
    return {"prompt": TASK_PROMPTS[task_setting][kind],
            "prompt_sha": hashlib.sha256(prompt.encode()).hexdigest()[:12],
            "model": model, "max_new_tokens": max_new_tokens, "max_chars": MAX_CHARS,
            "decoding": decoding}


def variant_name(setting: str, model: str, decoding: str) -> str:
    """Cache folder name; changes whenever a prompt, the model or the decoding changes."""
    p = TASK_PROMPTS[setting]
    key = read_prompt(p["doc"]) + read_prompt(p["query"]) + decoding
    return f"{setting}-{hashlib.sha256(key.encode()).hexdigest()[:6]}__{model.split('/')[-1]}"


CACHE_FILES = {"doc": "docs.jsonl", "query": "queries.jsonl"}


_ECHO = re.compile(r"\n\s*(Passage|Problem|Solution)\s*:.*", re.S)


def clean_profile(text: str) -> str:
    """The model sometimes appends the source text after its answer ("Passage: ..."); drop it.
    Applied on read, so the raw cache keeps exactly what the model generated."""
    return _ECHO.sub("", text).strip()


def _cache_paths(vdir: Path, kind: str) -> list[Path]:
    """Main cache file first, then any shard files left by parallel (one-per-GPU) runs."""
    stem = CACHE_FILES[kind].removesuffix(".jsonl")
    return [p for p in [vdir / CACHE_FILES[kind], *sorted(vdir.glob(f"{stem}.shard*.jsonl"))] if p.exists()]


def _read_rows(path: Path) -> dict[str, str]:
    rows = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                continue  # half-written last line from a killed run; that id is regenerated
            rows[d["_id"]] = d["profile"]
    return rows


def load_profiles(task: str, variant: str, kind: str, raw: bool = False) -> dict[str, str]:
    rows = {}
    for path in _cache_paths(PROFILES / task / variant, kind):
        rows.update(_read_rows(path))
    return rows if raw else {i: clean_profile(t) for i, t in rows.items()}


def _append(path: Path):
    """Open a cache file for appending, first ending any unterminated last line (e.g. a file
    written by Jupyter's %%writefile, or a run killed mid-write) so new rows start on their own line."""
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as f:
            f.seek(-1, 2)
            if f.read(1) != b"\n":
                with open(path, "a", encoding="utf-8") as g:
                    g.write("\n")
    return open(path, "a", encoding="utf-8")


def merge_shards(task: str, variant: str, kind: str) -> None:
    """Fold shard files into the main cache file (skipping ids already there), then delete them."""
    vdir = PROFILES / task / variant
    main = vdir / CACHE_FILES[kind]
    have = _read_rows(main) if main.exists() else {}
    shards = [p for p in _cache_paths(vdir, kind) if p != main]
    with _append(main) as f:
        for sp in shards:
            for i, text in _read_rows(sp).items():
                if i not in have:
                    have[i] = text
                    f.write(json.dumps({"_id": i, "profile": text}, ensure_ascii=False) + "\n")
    for sp in shards:
        sp.unlink()


def profile(task: Task, kind: str, ids: list[str], backend, variant: str, model: str,
            setting: str | None = None, max_new_tokens: int = 220, chunk: int = 256,
            shard: int | None = None) -> dict[str, str]:
    """Generate profiles for ids not yet cached; append to the cache as each chunk finishes.
    With shard=k, new rows go to <kind>.shard<k>.jsonl so parallel processes never share a file."""
    setting = setting or task.name
    vdir = PROFILES / task.name / variant
    vdir.mkdir(parents=True, exist_ok=True)
    meta_path = vdir / f"meta_{kind}.json"
    meta = _meta(setting, kind, model, max_new_tokens, backend.decoding)
    if meta_path.exists():
        old = json.loads(meta_path.read_text())
        assert old == meta, f"cache {vdir} was built with different settings:\n{old}\nvs\n{meta}"
    else:
        # atomic, since parallel shards may write it at the same moment
        tmp = meta_path.with_suffix(f".tmp{shard or 0}")
        tmp.write_text(json.dumps(meta, indent=1))
        tmp.replace(meta_path)

    done = load_profiles(task.name, variant, kind)
    todo = [i for i in ids if i not in done]
    prompts = build_prompts(task, kind, todo, setting)
    out = vdir / (CACHE_FILES[kind] if shard is None else CACHE_FILES[kind].replace(".jsonl", f".shard{shard}.jsonl"))
    with _append(out) as f:
        for c in range(0, len(todo), chunk):
            part = todo[c:c + chunk]
            for i, text in zip(part, backend.generate([prompts[i] for i in part])):
                done[i] = clean_profile(text)
                f.write(json.dumps({"_id": i, "profile": text}, ensure_ascii=False) + "\n")
            f.flush()
    return {i: done[i] for i in ids}


In [ ]:
%%writefile scripts/handtest_prompts.py
"""Week 1: hand-test profile prompts on 20 items per task (5 groups x 4).

A group is one query plus 3 of its gold documents, so items in a group share the hidden
property (same author / same proof move) and items in different groups don't. The report shows
every profile for reading, plus three checks:
  - length and format compliance (60-120 words; Math has the 3 fixed fields)
  - topic leakage: content words the profile copies from its source text
  - within-group vs across-group cosine similarity, for raw text and for profiles

    python scripts/handtest_prompts.py --task writing
    python scripts/handtest_prompts.py --task math --setting math_soln
    # Kaggle (2x T4): LLM on GPU 0 via vLLM, embedder on GPU 1
    python scripts/handtest_prompts.py --task writing --backend vllm --embed-device cuda:1
"""
import argparse
import gc
import random
import re
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from obliq.data import ROOT, load_task
from obliq.profiles import (TASK_PROMPTS, HFBackend, VLLMBackend, load_profiles, profile,
                            strip_math_query, variant_name)

MODELS = {"hf": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", "vllm": "Qwen/Qwen2.5-7B-Instruct-AWQ"}
EMBEDDER = "Qwen/Qwen3-Embedding-0.6B"
STOP = set("""this that with from have which their there they them then than when what where
would could should about into also only very such some more most other these those were been
being does doing your yours each both same just over under after before while because again
further once here many much will shall must might problem prove show find given let suppose
assume determine compute value values number numbers every exists exist""".split())


def sample_groups(task, n_groups=5, per_group=3, seed=0):
    rng = random.Random(seed)
    qids = sorted(task.qrels)
    rng.shuffle(qids)
    groups, used = [], set()
    for q in qids:
        rel = sorted(set(task.qrels[q]) - task.excluded.get(q, set()))
        if len(rel) < per_group or used & set(rel) or q in used:
            continue  # need disjoint groups so "different group" really means different property
        docs = rng.sample(rel, per_group)
        groups.append((q, docs))
        used |= set(task.qrels[q]) | {q}
        if len(groups) == n_groups:
            break
    return groups


def nwords(text):
    # counts "short_sentences,rhythmic" as 2 words, unlike str.split()
    return len(re.findall(r"[A-Za-z0-9']+", text))


def content_words(text):
    return {w for w in re.findall(r"[a-z]{4,}", text.lower()) if w not in STOP}


def p_at3(sim, labels):
    """Leave-one-out: share of each item's top-3 neighbours that are in its group (chance = 3/19)."""
    hits = []
    for i in range(len(labels)):
        top = [j for j in np.argsort(-sim[i]) if j != i][:3]
        hits.append(sum(labels[j] == labels[i] for j in top) / 3)
    return float(np.mean(hits))


def bm25_sim(texts):
    from rank_bm25 import BM25Okapi
    tok = lambda t: re.findall(r"\w+", t.lower())
    bm25 = BM25Okapi([tok(t) for t in texts])
    return np.array([bm25.get_scores(tok(t)) for t in texts])


def group_sims(emb, labels):
    sims = emb @ emb.T
    same = [sims[i, j] for i in range(len(labels)) for j in range(i + 1, len(labels)) if labels[i] == labels[j]]
    diff = [sims[i, j] for i in range(len(labels)) for j in range(i + 1, len(labels)) if labels[i] != labels[j]]
    return float(np.mean(same)), float(np.mean(diff))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", required=True, choices=["math", "writing"])
    ap.add_argument("--setting", default=None, help="math_soln = profile Math docs with solutions")
    ap.add_argument("--backend", default="hf", choices=["hf", "vllm"])
    ap.add_argument("--model", default=None)
    ap.add_argument("--embed-device", default="cuda:0")
    ap.add_argument("--rescore", action="store_true", help="use cached profiles, don't run the LLM")
    args = ap.parse_args()
    setting = args.setting or args.task
    args.model = args.model or MODELS[args.backend]

    task = load_task(args.task)
    groups = sample_groups(task)
    p = TASK_PROMPTS[setting]
    q_ids = [q for q, _ in groups]
    d_ids = [d for _, ds in groups for d in ds]
    backend_cls = HFBackend if args.backend == "hf" else VLLMBackend
    variant = variant_name(setting, args.model, backend_cls.decoding)
    if args.rescore:
        # re-score saved profiles (e.g. downloaded from Kaggle) without running the LLM
        q_prof = load_profiles(task.name, variant, "query")
        d_prof = load_profiles(task.name, variant, "doc")
        missing = [i for i in q_ids if i not in q_prof] + [i for i in d_ids if i not in d_prof]
        assert not missing, f"no cached profiles in {variant} for {missing[:5]}"
    else:
        backend = backend_cls(args.model)
        q_prof = profile(task, "query", q_ids, backend, variant, args.model, setting)
        d_prof = profile(task, "doc", d_ids, backend, variant, args.model, setting)
        del backend
        gc.collect()

    # items in display order: (group, kind, id, source text, profile)
    items = []
    for g, (q, docs) in enumerate(groups):
        src_q = strip_math_query(task.queries[q]) if args.task == "math" else task.queries[q]
        items.append((g, "query", q, src_q, q_prof[q]))
        items += [(g, "doc", d, task.corpus[d], d_prof[d]) for d in docs]

    import torch
    from sentence_transformers import SentenceTransformer
    torch.cuda.empty_cache()
    emb_model = SentenceTransformer(EMBEDDER, device=args.embed_device,
                                    model_kwargs={"torch_dtype": torch.float16})
    emb_model.max_seq_length = 2048
    labels = [it[0] for it in items]
    raw_texts, prof_texts = [it[3] for it in items], [it[4] for it in items]
    raw = emb_model.encode(raw_texts, normalize_embeddings=True, batch_size=4)
    prof = emb_model.encode(prof_texts, normalize_embeddings=True, batch_size=4)
    raw_same, raw_diff = group_sims(raw, labels)
    prof_same, prof_diff = group_sims(prof, labels)

    words = [nwords(it[4]) for it in items]
    leaks = [sorted(content_words(it[4]) & content_words(it[3])) for it in items]
    fields = {"math": ("Techniques:", "Proof move:", "Key insight:", "Also known as:"),
              "writing": ("Voice:", "Sentences:", "Punctuation:", "Tone and humour:",
                          "Signature phrases:", "Other habits:")}[args.task]
    fmt_ok = sum(all(f in it[4] for f in fields) for it in items)
    symbols = sum(bool(re.search(r"[$\\^_{}=]", it[4])) for it in items)

    lines = [f"# Prompt hand-test: {setting}", "",
             f"- Variant: `{variant}` (prompts: doc=`{p['doc']}`, query=`{p['query']}`)",
             f"- Model: `{args.model}`, decoding `{backend_cls.decoding}`",
             f"- Words per profile: min {min(words)}, median {int(np.median(words))}, max {max(words)} "
             f"(target 60-120); in range: {sum(60 <= w <= 120 for w in words)}/{len(words)}",
             f"- Format compliant ({len(fields)} fields): {fmt_ok}/{len(items)}",
             f"- Profiles containing formula symbols ($ \\ ^ _ {{ }} =): {symbols}/{len(items)}",
             f"- Profiles with copied content words: {sum(bool(l) for l in leaks)}/{len(items)}",
             "",
             "| Cosine (Qwen3-0.6B) | same group | different group | gap |",
             "|---|---|---|---|",
             f"| raw text | {raw_same:.3f} | {raw_diff:.3f} | {raw_same - raw_diff:+.3f} |",
             f"| profile | {prof_same:.3f} | {prof_diff:.3f} | {prof_same - prof_diff:+.3f} |",
             "", "A bigger gap means profiles separate the hidden property better than raw text does.", "",
             "| Leave-one-out P@3 (chance 0.158) | BM25 | dense |", "|---|---|---|",
             f"| raw text | {p_at3(bm25_sim(raw_texts), labels):.3f} | {p_at3(raw @ raw.T, labels):.3f} |",
             f"| profile | {p_at3(bm25_sim(prof_texts), labels):.3f} | {p_at3(prof @ prof.T, labels):.3f} |",
             "", "P@3 is closer to retrieval: for each item, how many of its top-3 neighbours share its "
             "hidden property. Cosine gap alone undersold BM25-friendly profiles (round 2).", ""]
    for (g, kind, i, src, pr), leak in zip(items, leaks):
        if kind == "query":
            lines += [f"## Group {g + 1}", ""]
        excerpt = src[:400].replace("\n", " ") + ("..." if len(src) > 400 else "")
        lines += [f"**{kind} `{i}`** ({nwords(pr)} words)", "",
                  f"> {excerpt}", "", f"Profile: {pr}", "",
                  f"Copied words: {', '.join(leak) if leak else '-'}", ""]
    out = ROOT / "reports" / f"handtest_{setting}.md"
    out.parent.mkdir(exist_ok=True)
    out.write_text("\n".join(lines), encoding="utf-8")
    print("\n".join(lines[:22]))
    print(f"\nreport: {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prompts/math_problem.txt
Read the math problem below. Do not solve it in full. Work out the core reasoning move a solution needs (the "aha moment") and describe it so that a mathematician would recognise the same move in a problem from a completely different field.
Name REASONING TECHNIQUES, not the topic. Do NOT mention the field or the objects involved (no words such as triangle, polynomial, matrix, prime, graph, integral, sequence). Only name a technique if a solution really needs it.
Write in words only: no formulas, symbols, LaTeX, or variable names.
Answer in exactly this format, 60-100 words in total:
Techniques: <2-4 technique names>
Proof move: <3-5 abstract steps joined by "->">
Key insight: <one abstract sentence>
Also known as: <2-3 other ways a mathematician might phrase the same move>

Problem:
{text}


In [ ]:
%%writefile prompts/math_problem_solution.txt
Read the math problem and its solution below. Identify the core reasoning move the solution uses (the "aha moment") and describe it so that a mathematician would recognise the same move in a problem from a completely different field.
Name REASONING TECHNIQUES, not the topic. Do NOT mention the field or the objects involved (no words such as triangle, polynomial, matrix, prime, graph, integral, sequence). Only name a technique the solution really uses.
Write in words only: no formulas, symbols, LaTeX, or variable names.
Answer in exactly this format, 60-100 words in total:
Techniques: <2-4 technique names>
Proof move: <3-5 abstract steps joined by "->">
Key insight: <one abstract sentence>
Also known as: <2-3 other ways a mathematician might phrase the same move>

Problem:
{problem}

Solution:
{solution}


In [ ]:
%%writefile prompts/writing_style.txt
You are a stylometry expert. Describe what makes the AUTHOR of the passage below recognisable, so that the same author could be identified from a passage on a completely different topic.
Focus on DISTINCTIVE habits. Skip traits that most blog posts share (such as "short sentences", "technical vocabulary", "informative tone", "uses commas") unless they are unusually strong here.
Do NOT mention the subject matter, names, places, products, or facts from the passage.
Answer in exactly this format, 60-100 words in total:
Voice: <person (I/we/you), stance toward the reader, formality>
Sentences: <typical length, rhythm, how sentences open and end>
Punctuation: <distinctive punctuation habits>
Tone and humour: <a few words>
Signature phrases: <2-4 short non-topical phrases or verbal tics quoted from the passage, such as hedges, intensifiers or transitions>
Other habits: <spelling, capitalisation, contractions, slang, formatting, paragraph structure>

Passage:
{text}


## Run the three hand-tests

LLM on GPU 0 (vLLM), embedder on GPU 1. Each run is a fresh process so vLLM releases GPU memory between runs.

In [ ]:
!python scripts/handtest_prompts.py --task writing --backend vllm --embed-device cuda:1

In [ ]:
!python scripts/handtest_prompts.py --task math --backend vllm --embed-device cuda:1

In [ ]:
!python scripts/handtest_prompts.py --task math --setting math_soln --backend vllm --embed-device cuda:1

In [ ]:
!zip -qr /kaggle/working/handtest_outputs.zip reports profiles
!ls -la /kaggle/working/handtest_outputs.zip